# Snapshot Creation and Profile Updates

This notebook processes trajectories and creates profile snapshots.

## What This Does

1. Extracts POI interactions from existing trajectories
2. Applies update formula: `P_new = 0.85 × P_old + 0.15 × O_trajectory`
3. Updates confidence using evidence-based formula
4. Creates snapshots after each trajectory
5. Handles discovered preferences (new POI categories)

**Update Process:**
- Get previous preferences (from initial profile or last snapshot)
- For each trajectory:
  - Extract POI interactions
  - Apply EWMA formula
  - Update confidence
  - Create snapshot
  - Store in triplestore

In [ ]:
import sys

import pandas as pd
from helpers.fuseki_config import query_endpoint, update_endpoint
from helpers.fuseki_queries import execute_sparql_query, format_results, load_query
from SPARQLWrapper import JSON, POST, SPARQLWrapper

# Import local helpers
sys.path.append('helpers')
from helpers.profile_learning import create_snapshot, format_preference_as_turtle, get_interaction_score

In [ ]:
# Configure SPARQL endpoints
print(f"Query endpoint: {query_endpoint}")
print(f"Update endpoint: {update_endpoint}")

sparql_query = SPARQLWrapper(query_endpoint)
sparql_update = SPARQLWrapper(update_endpoint)
sparql_update.method = POST

## Configuration

In [ ]:
def get_existing_users():
    """Fetch all users from the triplestore."""
    query = """
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?user ?userId
WHERE {
    ?user trajecto:formsTrajectory ?trajectory .
    ?user rdfs:label ?userId
}
ORDER BY ?userId
"""

    result = execute_sparql_query(query_endpoint, query)
    users = format_results(result)

    return users


users = get_existing_users()
print(f"\nFound {len(users)} users in the triplestore:\n")

if users.empty:
        print("❌ No users found! Make sure trajectories are loaded with user:belongsToUser property.")
else:
    display(users)

In [ ]:
# No longer need to manually select a user index
# The snapshot creation will loop through all users automatically
print(f"Total users to process: {len(users)}")

In [ ]:
def get_latest_snapshot(user_uri, verbose=False):
    """Get the latest snapshot for a user."""
    snapshot_query = load_query("get_latest_profile_snapshot.sparql")
    snapshot_query = snapshot_query.replace("?user_uri", f"<{user_uri}>")
    sparql_query.setQuery(snapshot_query)

    sparql_query.setReturnFormat(JSON)
    results = sparql_query.query().convert()

    snapshot_id = ""
    snapshot_date = ""
    preferences = []
    for binding in results['results']['bindings']:
        snapshot_id = binding['snapshot']['value']
        snapshot_date = binding['snapshotDate']['value']
        preferences.append({
                'category': binding['category']['value'],
                'score': binding['score']['value'],
                'confidence': binding['confidence']['value']
            })

    if verbose and preferences:
        display(pd.DataFrame(preferences))

    return snapshot_id, snapshot_date, preferences


def get_user_trajectories_after_snapshot(user_uri, last_snapshot_date):
    """Get trajectories for a user that occurred after the last snapshot."""
    check_trajectories_query = f"""
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX datacron: <http://www.datacron-project.eu/datAcron#>
PREFIX dul: <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?trajectory ?endTimestamp
WHERE {{
    <{user_uri}> trajecto:formsTrajectory ?trajectory .
    ?lastPos dul:isPartOf ?trajectory .
    ?lastPos trajecto:timestamp ?endTimestamp .
    FILTER NOT EXISTS {{
        ?lastPos trajecto:hasNextPosition ?anyNext .
    }}
    FILTER(?endTimestamp > "{last_snapshot_date}")
}}
ORDER BY ?endTimestamp
"""
    result = execute_sparql_query(query_endpoint, check_trajectories_query)
    return format_results(result)


def extract_trajectory_interactions(trajectory_uri, verbose=False):
    """Extract POI interactions from a trajectory."""
    interactions_query = load_query("get-trajectory-interactions.sparql")
    interactions_query = interactions_query.replace("?trajectory", f"<{trajectory_uri}>")
    sparql_query.setQuery(interactions_query)

    sparql_query.setReturnFormat(JSON)
    results = sparql_query.query().convert()

    interactions = []
    for binding in results['results']['bindings']:
        if 'interaction' in binding:
            interactions.append({
                'Interaction': binding['interaction']['value'],
                'POI': binding['poi_class']['value'],
                'Dominant': binding['isDominant']['value'] if 'isDominant' in binding else None,
                'Immersive': binding['isImmersive']['value'] if 'isImmersive' in binding else None
            })

    if verbose and interactions:
        display(pd.DataFrame(interactions))

    return interactions


print("Helper functions defined.")

In [ ]:
def process_user_snapshots(user_uri, user_id, verbose=False):
    """Process all pending trajectories for a user and create snapshots.

    Returns a dict with processing results.
    """
    results = {
        'user_id': user_id,
        'user_uri': user_uri,
        'snapshots_created': 0,
        'trajectories_processed': 0,
        'trajectories_without_interactions': 0,
        'skipped': False,
        'error': None
    }

    try:
        # Get latest snapshot
        last_snapshot, last_snapshot_date, last_snapshot_prefs = get_latest_snapshot(user_uri, verbose=verbose)

        if not last_snapshot:
            results['skipped'] = True
            results['error'] = "No initial snapshot found"
            return results

        # Get trajectories after last snapshot
        user_trajectories = get_user_trajectories_after_snapshot(user_uri, last_snapshot_date)

        if user_trajectories.empty:
            results['skipped'] = True
            results['error'] = "No new trajectories"
            return results

        # Transform preferences to dict
        current_prefs = {}
        for pref in last_snapshot_prefs:
            category = pref['category']
            current_prefs[category] = {
                'score': float(pref['score']),
                'confidence': float(pref['confidence']),
                'observation_count': 0
            }

        snapshots = []
        sorted_trajectories = user_trajectories.sort_values(by='endTimestamp')

        for _idx, row in sorted_trajectories.iterrows():
            trajectory_uri = row['trajectory']
            interactions_raw = extract_trajectory_interactions(trajectory_uri, verbose=verbose)

            if not interactions_raw:
                results['trajectories_without_interactions'] += 1
                if verbose:
                    print(f"      No interactions found for trajectory: {trajectory_uri}")
                continue

            # Transform interactions
            interactions = {}
            for interaction in interactions_raw:
                poi_category = interaction['POI']

                if interaction['Dominant'] == 'true':
                    interaction_type = 'DominantInteraction'
                elif interaction['Immersive'] == 'true':
                    interaction_type = 'ImmersiveInteraction'
                else:
                    interaction_type = 'MeaningfulInteraction'

                if poi_category not in interactions:
                    interactions[poi_category] = interaction_type
                else:
                    current_score = get_interaction_score(interactions[poi_category])
                    new_score = get_interaction_score(interaction_type)
                    if new_score > current_score:
                        interactions[poi_category] = interaction_type

            # Create snapshot
            snapshot = create_snapshot(
                user_id=user_id,
                trajectory_id=trajectory_uri,
                trajectory_end_time=row['endTimestamp'],
                interactions=interactions,
                previous_preferences=current_prefs
            )

            snapshots.append(snapshot)
            current_prefs = snapshot['preferences'].copy()
            results['trajectories_processed'] += 1

        # Store snapshots
        if snapshots:
            last_snapshot_num = int(last_snapshot.split('_snapshot_')[-1])

            for idx, snapshot in enumerate(snapshots, 1):
                snapshot_num = last_snapshot_num + idx
                snapshot_id = f"{user_uri}_snapshot_{snapshot_num}"

                # Check if exists
                check_query = f"""
PREFIX user: <http://w3id.org/trajecto/user#>
ASK {{ <{snapshot_id}> a user:ProfileSnapshot . }}
"""
                sparql_query.setQuery(check_query)
                sparql_query.setReturnFormat(JSON)
                exists = sparql_query.query().convert()['boolean']

                if not exists:
                    turtle_data = format_preference_as_turtle(snapshot_id, user_id, snapshot)
                    insert_query = f"""
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX trajecto: <http://w3id.org/trajecto#>

INSERT DATA {{
{turtle_data}
}}
"""
                    sparql_update.setQuery(insert_query)
                    sparql_update.query()
                    results['snapshots_created'] += 1
                elif verbose:
                    print(f"      Snapshot already exists: {snapshot_id}")

    except Exception as e:
        results['error'] = str(e)
        import traceback
        if verbose:
            traceback.print_exc()

    return results


print("Snapshot processing function defined.")

## Step 5: Process All Users

Loop through all users and create snapshots for their new trajectories.

In [ ]:
# Process all users
processing_results = []

print(f"Processing {len(users)} users...\n")
print("=" * 80)

for idx, row in users.iterrows():
    user_uri = row['user']
    user_id = row['userId']

    print(f"[{idx + 1}/{len(users)}] Processing user: {user_id}")

    result = process_user_snapshots(user_uri, user_id, verbose=False)
    processing_results.append(result)

    if result['skipped']:
        print(f"    ⏭️  Skipped: {result['error']}")
    elif result['error']:
        print(f"    ❌ Error: {result['error']}")
    else:
        traj_count = result['trajectories_processed']
        no_interact = result['trajectories_without_interactions']
        snap_count = result['snapshots_created']
        print(f"    ✅ Trajectories with interactions: {traj_count}, Without interactions: {no_interact}, Snapshots created: {snap_count}")

print("=" * 80)
print("\nProcessing complete!")

## Step 6: Processing Summary

View a summary of the snapshot creation results.

In [ ]:
# Summary statistics
results_df = pd.DataFrame(processing_results)

total_snapshots = results_df['snapshots_created'].sum()
total_trajectories = results_df['trajectories_processed'].sum()
total_without_interactions = results_df['trajectories_without_interactions'].sum()
users_with_snapshots = (results_df['snapshots_created'] > 0).sum()
skipped = results_df['skipped'].sum()
errors = ((results_df['error'].notna()) & (~results_df['skipped'])).sum()

print("=" * 80)
print("PROCESSING SUMMARY")
print("=" * 80)
print(f"\nTotal users processed: {len(users)}")
print(f"  ✅ Users with new snapshots: {users_with_snapshots}")
print(f"  ✅ Total snapshots created:  {total_snapshots}")
print(f"  ✅ Trajectories with interactions: {total_trajectories}")
print(f"  ⚠️  Trajectories without interactions: {total_without_interactions}")
print(f"  ⏭️  Skipped (no profile/trajectories): {skipped}")
print(f"  ❌ Errors: {errors}")

# Show users that had snapshots created
if users_with_snapshots > 0:
    print("\nUsers with new snapshots:")
    active_users = results_df[results_df['snapshots_created'] > 0][['user_id', 'trajectories_processed', 'snapshots_created']]
    display(active_users)

# Show users with trajectories but no interactions
users_no_interact = results_df[(results_df['trajectories_without_interactions'] > 0) & (results_df['trajectories_processed'] == 0)]
if len(users_no_interact) > 0:
    print("\nUsers with trajectories but NO interactions (data issue?):")
    display(users_no_interact[['user_id', 'trajectories_without_interactions']])

# Show errors if any
if errors > 0:
    print("\nUsers with errors:")
    error_users = results_df[(results_df['error'].notna()) & (~results_df['skipped'])][['user_id', 'error']]
    display(error_users)

print("\n" + "=" * 80)

## Step 7: Verify All Snapshots

Query the triplestore to verify snapshots for all users.

In [ ]:
# Verify all snapshots across all users
verify_all_query = """
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?userId (COUNT(DISTINCT ?snapshot) as ?snapshotCount)
WHERE {
    ?user rdfs:label ?userId .
    ?user user:hasProfile ?profile .
    ?profile user:hasSnapshot ?snapshot .
}
GROUP BY ?userId
ORDER BY ?userId
"""

result = execute_sparql_query(query_endpoint, verify_all_query)
verification_df = format_results(result)

print("\nSnapshot Verification - All Users:")
print("=" * 80)

if not verification_df.empty:
    display(verification_df)
    total_snapshots_in_store = verification_df['snapshotCount'].astype(int).sum()
    print(f"\n✅ Total snapshots in triplestore: {total_snapshots_in_store}")
else:
    print("❌ No snapshots found in triplestore!")

## (Optional) View Details for a Specific User

Use the cells below to inspect snapshots and preferences for a specific user.

In [ ]:
# Change USER_INDEX to inspect a different user
USER_INDEX = 0  # Change this to view a different user

if not users.empty and len(users) > USER_INDEX:
    USER_URI = users.iloc[USER_INDEX]['user']
    USER_ID = users.iloc[USER_INDEX]['userId']

    verify_query = f"""
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?snapshot ?timestamp (COUNT(?pref) as ?prefCount)
WHERE {{
    <{USER_URI}> user:hasProfile ?profile .
    ?profile user:hasSnapshot ?snapshot .
    ?snapshot user:updateTimestamp ?timestamp ;
              user:hasPreferences ?pref .
}}
GROUP BY ?snapshot ?timestamp
ORDER BY ?timestamp
"""

    sparql_query.setQuery(verify_query)
    sparql_query.setReturnFormat(JSON)
    results = sparql_query.query().convert()

    snapshot_data = []
    for binding in results['results']['bindings']:
        snapshot_data.append({
            'Snapshot': binding['snapshot']['value'].split('#')[-1],
            'Timestamp': binding['timestamp']['value'],
            'Preferences': int(binding['prefCount']['value'])
        })

    print(f"Snapshots for user: {USER_ID}\n")
    if snapshot_data:
        display(pd.DataFrame(snapshot_data))
        print(f"\n✅ Total snapshots: {len(snapshot_data)}")
    else:
        print("No snapshots found for this user.")
else:
    print("Invalid USER_INDEX or no users available.")

## (Optional) Preference Evolution for a Specific User

View how preferences changed over time for a specific user.

In [ ]:
# Query preference evolution for the selected user (uses USER_INDEX from above)
if not users.empty and len(users) > USER_INDEX:
    USER_URI = users.iloc[USER_INDEX]['user']
    USER_ID = users.iloc[USER_INDEX]['userId']

    evolution_query = f"""
PREFIX user: <http://w3id.org/trajecto/user#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?timestamp ?category ?score ?confidence ?obsCount
WHERE {{
    <{USER_URI}> user:hasProfile ?profile .
    ?profile user:hasSnapshot ?snapshot .
    ?snapshot user:updateTimestamp ?timestamp ;
              user:hasPreferences ?pref .

    ?pref user:prefersPoiCategory ?category ;
          user:preferenceScore ?score ;
          user:confidenceLevel ?confidence ;
          user:observationCount ?obsCount .
}}
ORDER BY ?timestamp ?category
"""

    sparql_query.setQuery(evolution_query)
    sparql_query.setReturnFormat(JSON)
    results = sparql_query.query().convert()

    evolution_data = []
    for binding in results['results']['bindings']:
        evolution_data.append({
            'Timestamp': binding['timestamp']['value'],
            'Category': binding['category']['value'],
            'Score': float(binding['score']['value']),
            'Confidence': float(binding['confidence']['value']),
            'Observations': int(binding['obsCount']['value'])
        })

    if evolution_data:
        df_evolution = pd.DataFrame(evolution_data)

        print(f"Preference Evolution for user: {USER_ID}")
        print("=" * 80)

        for category in df_evolution['Category'].unique():
            cat_data = df_evolution[df_evolution['Category'] == category]
            print(f"\n{category}:")
            print(f"  Initial Score: {cat_data['Score'].iloc[0]:.3f}")
            print(f"  Final Score: {cat_data['Score'].iloc[-1]:.3f}")
            print(f"  Delta: {cat_data['Score'].iloc[-1] - cat_data['Score'].iloc[0]:+.3f}")
            print(f"  Confidence: {cat_data['Confidence'].iloc[0]:.3f} → {cat_data['Confidence'].iloc[-1]:.3f}")
            print(f"  Total Observations: {cat_data['Observations'].iloc[-1]}")

        print("\n" + "=" * 80)
    else:
        print(f"No evolution data found for user: {USER_ID}")
else:
    print("Invalid USER_INDEX or no users available.")

## Summary

**Snapshot Creation Complete!**

- Processed all users with trajectories
- Applied EWMA formula: P_new = 0.85 × P_old + 0.15 × O_trajectory
- Updated confidence using evidence-based formula
- Handled discovered preferences (new categories)
- Stored all snapshots in triplestore

**Next Steps:**
1. Run preference visualization notebooks to see how preferences evolved
2. Use the optional cells above to inspect specific users

In [ ]:
# Validate KG state against stage-08-user-profiles SHACL shapes
from helpers.shacl_validator import validate_before_import, construct_validation_graph

proposed = construct_validation_graph('stage-08-user-profiles')
print(f"Fetched {len(proposed)} triples for validation")
report = validate_before_import(proposed, 'stage-08-user-profiles', halt_on_violations=True)